In [2]:
import pandas as pd
import numpy as np
from prophet import Prophet
import plotly.express as px
import plotly.graph_objects as go
from sklearn.metrics import mean_absolute_error, mean_squared_error
import warnings
warnings.filterwarnings('ignore')



In [3]:
" loading new dataset"
df = pd.read_csv('../data/walmart_enriched.csv')
df['Date'] = pd.to_datetime(df['Date'])

print("Shape:", df.shape)
print("Date range:", df['Date'].min(), "to", df['Date'].max())


Shape: (420212, 24)
Date range: 2010-02-05 00:00:00 to 2012-10-26 00:00:00


In [21]:
# Filter for Store 1, Department 81 (most stable)
store1_dept81 = df[(df['Store'] == 1) & (df['Dept'] == 81)].copy()

# Sort by date
store1_dept81 = store1_dept81.sort_values('Date').reset_index(drop=True)

print("Total weeks of data:", len(store1_dept81))
print("Date range:", store1_dept81['Date'].min(), "to", store1_dept81['Date'].max())
print("\nSales stats:")
print(store1_dept81['Weekly_Sales'].describe().round(2))

Total weeks of data: 143
Date range: 2010-02-05 00:00:00 to 2012-10-26 00:00:00

Sales stats:
count      143.00
mean     29780.70
std       1849.37
min      24041.67
25%      28531.85
50%      29979.34
75%      30943.00
max      36081.52
Name: Weekly_Sales, dtype: float64


In [22]:
'''preparing data for the prophet model
    the model needs two cols one '''
prophet_df = store1_dept81[['Date', 'Weekly_Sales']].rename(
    columns={
        'Date': 'ds',
        'Weekly_Sales': 'y'
    }
)

print("Prophet dataframe ready ✅")
print(prophet_df.head())

Prophet dataframe ready ✅
          ds         y
0 2010-02-05  30052.75
1 2010-02-12  30694.25
2 2010-02-19  28784.91
3 2010-02-26  28213.79
4 2010-03-05  29957.47


In [23]:
"splitting data: 80% for training and 20% for testing"

split_index = int(len(prophet_df) * 0.8)

train = prophet_df.iloc[:split_index]
test  = prophet_df.iloc[split_index:]

print("Training rows  :", len(train))
print("Testing rows   :", len(test))
print("Training until :", train['ds'].max())
print("Testing from   :", test['ds'].min())

Training rows  : 114
Testing rows   : 29
Training until : 2012-04-06 00:00:00
Testing from   : 2012-04-13 00:00:00


In [24]:
"training model"

model = Prophet(
    yearly_seasonality  = True,   # learns yearly patterns
    weekly_seasonality  = True,   # learns weekly patterns
    daily_seasonality   = False,  # we have weekly data so False
    seasonality_mode    = 'multiplicative',  
    interval_width      = 0.95    # 95% confidence interval
)

# Add US holidays (built into Prophet)
model.add_country_holidays(country_name='US')

# Train the model
model.fit(train)

print("Prophet model trained successfully")

18:48:36 - cmdstanpy - INFO - Chain [1] start processing
18:48:36 - cmdstanpy - INFO - Chain [1] done processing


Prophet model trained successfully


In [25]:
# Creating future dataframe
# We predict for the test period + 12 more weeks into the future
future = model.make_future_dataframe(
    periods = len(test) + 12,
    freq    = 'W'  # W means weekly
)

# Generating forecast
forecast = model.predict(future)

print("Forecast generated ")
print("Forecast columns:", ['ds', 'yhat', 'yhat_lower', 'yhat_upper'])
print("\nSample forecast:")
forecast[['ds', 'yhat', 'yhat_lower', 'yhat_upper']].tail(15)

Forecast generated 
Forecast columns: ['ds', 'yhat', 'yhat_lower', 'yhat_upper']

Sample forecast:


,ds,yhat,yhat_lower,yhat_upper
140,2012-10-07,31817.808020,29331.977979,34393.311509
141,2012-10-14,31713.021929,29030.938490,34502.202163
142,2012-10-21,31050.907978,28445.276458,33409.365946
143,2012-10-28,30584.260094,28048.601636,33281.993329
144,2012-11-04,30989.735181,28035.918259,33591.847339
145,2012-11-11,32428.230760,29849.192260,35143.842393
146,2012-11-18,32846.522001,30451.507902,35458.853831
147,2012-11-25,32662.584897,30126.772290,35308.582287
148,2012-12-02,31803.770912,29295.918521,34450.486524
149,2012-12-09,31190.720869,28683.936285,33916.222625


In [26]:
# Merge forecast with actual test values
forecast_test = forecast[['ds', 'yhat', 'yhat_lower', 'yhat_upper']].copy()
forecast_test = forecast_test.merge(prophet_df, on='ds', how='left')

# Plot
fig = go.Figure()

# Actual sales
fig.add_trace(go.Scatter(
    x=prophet_df['ds'],
    y=prophet_df['y'],
    mode='lines',
    name='Actual Sales',
    line=dict(color='#2E86AB', width=2)
))

# Predicted sales
fig.add_trace(go.Scatter(
    x=forecast['ds'],
    y=forecast['yhat'],
    mode='lines',
    name='Predicted Sales',
    line=dict(color='#E84855', width=2, dash='dash')
))

# Confidence interval (the range)
fig.add_trace(go.Scatter(
    x=pd.concat([forecast['ds'], forecast['ds'][::-1]]),
    y=pd.concat([forecast['yhat_upper'], forecast['yhat_lower'][::-1]]),
    fill='toself',
    fillcolor='rgba(232, 72, 85, 0.1)',
    line=dict(color='rgba(255,255,255,0)'),
    name='95% Confidence Interval'
))

fig.update_layout(
    title='Store 1 Dept 1 — Sales Forecast (Prophet)',
    xaxis_title='Date',
    yaxis_title='Weekly Sales ($)',
    hovermode='x unified'
)

fig.show()

In [27]:
# Debug cell - let's see exactly what's happening
print("Test dates sample:")
print(test['ds'].head())
print("Type:", test['ds'].dtype)

print("\nForecast dates sample:")
print(forecast['ds'].head())
print("Type:", forecast['ds'].dtype)

print("\nDo any dates match?")
common = set(pd.to_datetime(test['ds']).dt.date) & set(pd.to_datetime(forecast['ds']).dt.date)
print("Common dates found:", len(common))
print("Sample common dates:", list(common)[:5])

Test dates sample:
114   2012-04-13
115   2012-04-20
116   2012-04-27
117   2012-05-04
118   2012-05-11
Name: ds, dtype: datetime64[ns]
Type: datetime64[ns]

Forecast dates sample:
0   2010-02-05
1   2010-02-12
2   2010-02-19
3   2010-02-26
4   2010-03-05
Name: ds, dtype: datetime64[ns]
Type: datetime64[ns]

Do any dates match?
Common dates found: 0
Sample common dates: []


In [28]:
# Check exact date values
print("Test dates (first 3):", test['ds'].dt.date.tolist()[:3])
print("Forecast dates (first 3):", forecast['ds'].dt.date.tolist()[:3])

# Fix - round both to week level and match
test_forecast = forecast[['ds', 'yhat', 'yhat_lower', 'yhat_upper']].copy()

# Normalize both to same frequency by rounding to nearest week
test_copy = test.copy()
test_copy['ds_week'] = pd.to_datetime(test_copy['ds']).dt.to_period('W').dt.start_time
test_forecast['ds_week'] = pd.to_datetime(test_forecast['ds']).dt.to_period('W').dt.start_time

# Merge on week period instead of exact date
results = test_copy.merge(test_forecast, on='ds_week', how='inner')

print("\nMerged rows:", len(results))
print(results[['ds_week', 'y', 'yhat']].head())

Test dates (first 3): [datetime.date(2012, 4, 13), datetime.date(2012, 4, 20), datetime.date(2012, 4, 27)]
Forecast dates (first 3): [datetime.date(2010, 2, 5), datetime.date(2010, 2, 12), datetime.date(2010, 2, 19)]

Merged rows: 29
     ds_week         y          yhat
0 2012-04-09  31857.92  32176.383492
1 2012-04-16  30648.90  31944.204601
2 2012-04-23  30244.48  31875.838429
3 2012-04-30  32261.06  32052.572017
4 2012-05-07  31571.96  32328.557917


In [29]:
# Calculate error metrics on the merged results
mae  = mean_absolute_error(results['y'], results['yhat'])
rmse = np.sqrt(mean_squared_error(results['y'], results['yhat']))
mape = (abs((results['y'] - results['yhat']) / results['y']).mean()) * 100

print("=" * 40)
print("        MODEL ACCURACY RESULTS")
print("=" * 40)
print(f"  MAE  (Mean Absolute Error)  : ${mae:,.2f}")
print(f"  RMSE (Root Mean Sq Error)   : ${rmse:,.2f}")
print(f"  MAPE (Mean Abs % Error)     : {mape:.2f}%")
print(f"  Accuracy                    : {100 - mape:.2f}%")
print("=" * 40)

        MODEL ACCURACY RESULTS
  MAE  (Mean Absolute Error)  : $1,291.25
  RMSE (Root Mean Sq Error)   : $1,614.48
  MAPE (Mean Abs % Error)     : 4.16%
  Accuracy                    : 95.84%


In [30]:
# Let's see the actual sales pattern of Dept 1
import plotly.express as px

fig = px.line(
    prophet_df,
    x='ds',
    y='y',
    title='Store 1 Dept 1 — Actual Sales Pattern'
)
fig.show()

print("Sales stats:")
print(prophet_df['y'].describe().round(2))


Sales stats:
count      143.00
mean     29780.70
std       1849.37
min      24041.67
25%      28531.85
50%      29979.34
75%      30943.00
max      36081.52
Name: y, dtype: float64


In [31]:
# Find most stable department in Store 1
store1 = df[df['Store'] == 1].copy()

dept_stats = store1.groupby('Dept')['Weekly_Sales'].agg([
    'mean',
    'std',
    'count'
]).reset_index()

# Coefficient of variation (lower = more stable)
dept_stats['cv'] = dept_stats['std'] / dept_stats['mean']

# Only keep departments with full data
dept_stats = dept_stats[dept_stats['count'] >= 130]

# Sort by most stable
dept_stats = dept_stats.sort_values('cv')

print("Top 5 most stable departments in Store 1:")
print(dept_stats.head())

Top 5 most stable departments in Store 1:
    Dept          mean          std  count        cv
62  81.0  29780.696573  1849.372057    143  0.062100
12  13.0  38692.880490  2509.848052    143  0.064866
7    8.0  35718.257622  2490.769188    143  0.069734
37  40.0  58510.409161  4264.078123    143  0.072877
1    2.0  46102.090420  3440.673222    143  0.074632


In [32]:
# Get next 12 weeks forecast
future_forecast = forecast[forecast['ds'] > df['Date'].max()][
    ['ds', 'yhat', 'yhat_lower', 'yhat_upper']
].head(12)

# Business cost assumptions
holding_cost_per_unit  = 2   # cost to hold 1 extra unit
stockout_cost_per_unit = 8   # cost of missing 1 sale
avg_price_per_unit     = 50  # average product price

# Calculate recommended order quantity
future_forecast = future_forecast.copy()
future_forecast['Predicted_Units']    = (future_forecast['yhat'] / avg_price_per_unit).astype(int)
future_forecast['Min_Order_Units']    = (future_forecast['yhat_lower'] / avg_price_per_unit).astype(int)
future_forecast['Max_Order_Units']    = (future_forecast['yhat_upper'] / avg_price_per_unit).astype(int)
future_forecast['Recommended_Order']  = (future_forecast['Predicted_Units'] * 1.03).astype(int)

# Cost calculations
future_forecast['Overstock_Cost']  = (
    (future_forecast['Recommended_Order'] - future_forecast['Predicted_Units'])
    * holding_cost_per_unit
)
future_forecast['Stockout_Cost']   = (
    (future_forecast['Predicted_Units'] - future_forecast['Min_Order_Units'])
    * stockout_cost_per_unit
)

print("📦 INVENTORY RECOMMENDATIONS — NEXT 12 WEEKS")
print("=" * 70)
print(future_forecast[[
    'ds',
    'Predicted_Units',
    'Min_Order_Units',
    'Recommended_Order',
    'Max_Order_Units',
    'Overstock_Cost',
    'Stockout_Cost'
]].to_string(index=False))

📦 INVENTORY RECOMMENDATIONS — NEXT 12 WEEKS
        ds  Predicted_Units  Min_Order_Units  Recommended_Order  Max_Order_Units  Overstock_Cost  Stockout_Cost
2012-10-28              611              560                629              665              36            408
2012-11-04              619              560                637              671              36            472
2012-11-11              648              596                667              702              38            416
2012-11-18              656              609                675              709              38            376
2012-11-25              653              602                672              706              38            408
2012-12-02              636              585                655              689              38            408
2012-12-09              623              573                641              678              36            400
2012-12-16              626              571                

In [34]:
future_forecast.to_csv('../data/forecast_results.csv', index=False)
print(" Forecast results saved")

forecast.to_csv('../data/prophet_forecast_full.csv', index=False)
print(" Full forecast saved")

 Forecast results saved
 Full forecast saved
